In [20]:
import csv
from datetime import datetime
from pathlib import Path
from urllib.parse import quote

import requests


WEATHER_URL = "https://wttr.in/"
REQUEST_TIMEOUT = 10


def get_weather(city_name):
    """Fetch and normalize current weather data for one city."""
    city_name = city_name.strip()
    if not city_name:
        raise ValueError("City name cannot be empty.")

    url = f"{WEATHER_URL}{quote(city_name, safe='')}?format=j1"
    try:
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        current = response.json()["current_condition"][0]
        return {
            "city": city_name,
            "temp_C": int(current["temp_C"]),
            "FeelsLikeC": int(current["FeelsLikeC"]),
            "humidity": int(current["humidity"]),
            "windspeedKmph": int(current["windspeedKmph"]),
            "description": current["weatherDesc"][0]["value"],
        }
    except requests.exceptions.Timeout:
        print(f"Weather request for {city_name} timed out. Check your connection and try again.")
    except requests.exceptions.RequestException as error:
        print(f"Could not fetch weather for {city_name}: {error}")
    except (KeyError, IndexError, TypeError, ValueError) as error:
        print(f"Unexpected weather data for {city_name}: {error}")
    return None


def print_weather(weather):
    """Print a weather record, including wind and the heat warning."""
    print(f"📍 {weather['city']} weather:")
    print(f"Temperature: {weather['temp_C']}°C (feels like: {weather['FeelsLikeC']}°C)")
    print(f"Humidity: {weather['humidity']}%")
    print(f"Wind speed: {weather['windspeedKmph']} km/h")
    print(f"Conditions: {weather['description']}")
    if weather["temp_C"] > 35:
        print("⚠️ It's hot today!")

## Fetch live weather for one city

In [21]:
city = input("Shehar ka naam likho (e.g. Lahore): ").strip()
if city:
    result = get_weather(city)
    if result:
        print_weather(result)
else:
    print("Please enter a city name.")

📍 Islamabad weather:
Temperature: 24°C (feels like: 22°C)
Humidity: 37%
Wind speed: 9 km/h
Conditions: Smoky haze


## Compare temperatures

In [22]:
def compare_cities(city1, city2):
    """Fetch two cities and print which one currently has the higher temperature."""
    first = get_weather(city1)
    second = get_weather(city2)
    if first is None or second is None:
        print("Could not compare cities because weather data was unavailable.")
        return None

    if first["temp_C"] > second["temp_C"]:
        warmer = first
    elif second["temp_C"] > first["temp_C"]:
        warmer = second
    else:
        print(f"{city1} and {city2} are equally warm at {first['temp_C']}°C.")
        return None

    print(
        f"{warmer['city']} is warmer: {warmer['temp_C']}°C "
        f"(compared with {first['temp_C'] if warmer is second else second['temp_C']}°C)."
    )
    return warmer["city"]


# Try it with two cities, for example: compare_cities("Lahore", "Karachi")

In [23]:
# get_weather() reports timeouts, request failures, and unexpected API responses.
# It returns None for those fetch/data errors, so callers can continue with other cities.

## Try the comparison function

In [24]:
# Example usage: compare_cities("Lahore", "Karachi")

# All cities: print weather and save it to CSV

In [25]:
cities = ["Lahore", "Islamabad", "Karachi", "Multan"]
csv_path = Path("weather_log.csv")
has_header = csv_path.exists() and csv_path.stat().st_size > 0

with csv_path.open(mode="a", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    if not has_header:
        writer.writerow([
            "fetched_at", "city", "temp_C", "FeelsLikeC", "humidity",
            "windspeedKmph", "description"
        ])

    for city_name in cities:
        weather = get_weather(city_name)
        if weather is not None:
            print_weather(weather)
            writer.writerow([
                datetime.now().isoformat(timespec="seconds"),
                weather["city"],
                weather["temp_C"],
                weather["FeelsLikeC"],
                weather["humidity"],
                weather["windspeedKmph"],
                weather["description"],
            ])

print(f"Weather log updated: {csv_path.resolve()}")

📍 Lahore weather:
Temperature: 34°C (feels like: 31°C)
Humidity: 20%
Wind speed: 10 km/h
Conditions: Clear 
📍 Islamabad weather:
Temperature: 24°C (feels like: 22°C)
Humidity: 37%
Wind speed: 9 km/h
Conditions: Smoky haze
📍 Karachi weather:
Temperature: 27°C (feels like: 31°C)
Humidity: 83%
Wind speed: 22 km/h
Conditions: Patchy rain nearby
📍 Multan weather:
Temperature: 38°C (feels like: 34°C)
Humidity: 17%
Wind speed: 18 km/h
Conditions: Clear 
⚠️ It's hot today!
Weather log updated: D:\Adv Python dev\Live Weather Fetcher\weather_log.csv
